# LangChain v1 — the building blocks

**Week 12 · Session 1**

Two weeks ago you wrote an agent loop by hand. Last week the OpenAI Agents SDK packaged
that loop. This week: **LangChain**, the most widely used library for building on LLMs.

LangChain is bigger than the Agents SDK, and it isn't mainly an agent framework. It's a
**library of interchangeable parts**: models from any provider, prompt templates,
structured output, retrievers, tools, and a way to snap them together (LCEL). Agents
(`create_agent`, next session) are one thing you can build from the parts.

**Today: the parts, then one real project with no agent at all** — a pipeline that reads
a CSV of product reviews and writes a report. By the end you'll know when you *don't*
need an agent, which is most of the time.

> ### ⚠️ If you learned LangChain before late 2025
> LangChain **v1** removed a lot. `LLMChain`, `SequentialChain`, `RetrievalQA`,
> `ConversationBufferMemory` and `AgentExecutor` are **gone** from `langchain` — imports
> fail. They live on in `langchain-classic` for old code. Most tutorials online still use
> them, so this notebook shows what replaced each one.

In [1]:
%pip install -q "langchain==1.4.2" "langchain-openai==1.6.2" python-dotenv pandas


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, time, getpass
from pathlib import Path

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True) or Path.cwd().parent / ".env")
except ImportError:
    pass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

# Anaconda 'base' fix (same as Week 11). httpx2 2.12+ decompresses brotli with output_buffer_limit=... as a keyword.
# Anaconda's brotli 1.x exposes process() as a C function that rejects kwargs
# ("TypeError: process() takes no keyword arguments"). Wrap it so OpenAI
# Responses (which often arrive Content-Encoding: br) can be decoded.
try:
    import brotli as _brotli
    from httpx2._decoders import BrotliDecoder as _BrotliDecoder
    _probe = _brotli.Decompressor()
    try:
        _probe.process(b"", output_buffer_limit=1)
    except TypeError:
        _brotli_init = _BrotliDecoder.__init__
        def _brotli_init_compat(self, *args, **kwargs):
            _brotli_init(self, *args, **kwargs)
            _impl = self._decompress
            self._decompress = lambda data, output_buffer_limit=None, **_k: _impl(data)
        _BrotliDecoder.__init__ = _brotli_init_compat
except Exception:
    pass

import langchain
print("langchain", langchain.__version__)

langchain 1.4.2


---
# Part 1 — Models: one interface, any provider

This is the first reason people pick LangChain. `init_chat_model` takes a
`"provider:model"` string and returns a chat model with the **same interface** whatever
the provider. Switching from OpenAI to Anthropic, Google, Mistral or a local Ollama model
is a one-string change (plus that provider's package and key).

In [3]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

reply = llm.invoke("In one sentence, what is LangChain?")
print(type(reply).__name__)
print(reply.content)
print("\ntokens:", reply.usage_metadata)

AIMessage
LangChain is a framework designed to simplify the development of applications that integrate large language models with external data sources and tools.

tokens: {'input_tokens': 16, 'output_tokens': 24, 'total_tokens': 40, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


`invoke` returns an `AIMessage`, not a string: content plus metadata such as token
usage. The same model could be:

```python
init_chat_model("anthropic:claude-sonnet-5")     # pip install langchain-anthropic
init_chat_model("google_genai:gemini-2.5-flash")   # pip install langchain-google-genai
init_chat_model("ollama:llama3.2")                 # a local model, no key at all
```

Nothing else in this notebook would change. Compare that with last week's SDK, which is
OpenAI-first.

## Messages

Chat models take a list of messages — the same `system` / `user` / `assistant` roles you
used in Week 10, as classes.

In [4]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage("You are a terse product-support assistant for an Indian electronics brand."),
    HumanMessage("My earbuds won't pair with my phone."),
    AIMessage("Put both buds in the case, hold the button 10 seconds, then pair again."),
    HumanMessage("Tried that. Still nothing."),
]
print(llm.invoke(messages).content)

Ensure Bluetooth is on and no other device is connected. Restart your phone and try pairing again. If issue persists, reset earbuds by holding button 15 seconds until LED flashes red-blue. Then pair again.


## Every component runs the same four ways

Every LangChain component — a model, a prompt, a whole pipeline — is a **Runnable** with
the same methods: `invoke` (one input), `batch` (many, in parallel), `stream` (token by
token), plus async versions (`ainvoke`, `abatch`, `astream`).

In [5]:
# batch: many inputs, run concurrently
t0 = time.time()
answers = llm.batch(["Capital of France?", "Capital of Japan?", "Capital of Kenya?"],
                    config={"max_concurrency": 3})
print([a.content for a in answers], f"in {time.time() - t0:.1f}s")

# stream: tokens as they arrive
for chunk in llm.stream("Count from 1 to 8, comma separated."):
    print(chunk.content, end="", flush=True)
print()

['The capital of France is Paris.', 'The capital of Japan is Tokyo.', 'The capital of Kenya is Nairobi.'] in 0.9s


1

,

2

,

...

6

,

7

,

8

---
# Part 2 — Prompt templates

A prompt template is a prompt with **named holes**. You fill them at run time, so one
template serves every input — and it's the unit you version, test and reuse.

In [6]:
from langchain_core.prompts import ChatPromptTemplate

summarise = ChatPromptTemplate.from_messages([
    ("system", "You summarise customer reviews for a {audience}. Use at most {words} words."),
    ("human", "Review: {review}"),
])

print(summarise.input_variables)
filled = summarise.invoke({"audience": "product manager", "words": 15,
                           "review": "Battery is great but the app keeps logging me out."})
for m in filled.messages:
    print(f"{m.type:>6}: {m.content}")

['audience', 'review', 'words']
system: You summarise customer reviews for a product manager. Use at most 15 words.
 human: Review: Battery is great but the app keeps logging me out.


Two more features you'll use constantly:

- **`partial`** fixes some variables up front, leaving a template that needs fewer inputs.
- **`MessagesPlaceholder`** leaves a slot for a whole *list* of messages — conversation
  history. This is how memory gets into a prompt (the job `ConversationBufferMemory` used
  to do).

In [7]:
from langchain_core.prompts import MessagesPlaceholder

for_pm = summarise.partial(audience="product manager", words=12)
print("still needs:", for_pm.input_variables)

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful support assistant."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
filled = chat_prompt.invoke({
    "history": [HumanMessage("My watch is the Orbit."), AIMessage("Got it — the Nimbus Orbit.")],
    "question": "How do I reset it?",
})
print([m.type for m in filled.messages])

still needs: ['review']
['system', 'human', 'ai', 'human']


---
# Part 3 — Structured output

In Week 11 you gave the agent an `output_type`. LangChain's equivalent works on **any**
model: `with_structured_output(PydanticModel)`. You get a validated Python object back.

(You'll see older code use *output parsers* — `PydanticOutputParser` plus "format
instructions" pasted into the prompt. `with_structured_output` replaced that for any model
that supports tool calling or JSON mode, which is nearly all of them.)

In [8]:
from typing import Literal
from pydantic import BaseModel, Field


class ReviewAnalysis(BaseModel):
    sentiment: Literal["positive", "negative", "mixed"]
    issues: list[str] = Field(description="Concrete product problems, each 2-5 words. Empty if none.")
    category: Literal["battery", "sound", "connectivity", "build quality", "comfort",
                      "app", "delivery", "support", "value", "other"] = Field(
        description="The single main topic of the review")
    severity: int = Field(ge=1, le=5, description="1 = minor gripe, 5 = product unusable or safety issue")
    language: Literal["english", "hinglish", "hindi"]


analyst = llm.with_structured_output(ReviewAnalysis)
result = analyst.invoke("Sound quality mast hai but charging case ka hinge 2 weeks mein toot gaya. Bekaar.")
print(type(result).__name__)
print(result.model_dump_json(indent=2))

ReviewAnalysis
{
  "sentiment": "mixed",
  "issues": [
    "charging case hinge broken"
  ],
  "category": "build quality",
  "severity": 3,
  "language": "hinglish"
}


It read a **Hinglish** review and returned typed fields. The `Literal` types constrain the
answer to your categories, and `Field(ge=1, le=5)` is validated — this is data your code
can aggregate. The descriptions in `Field(...)` are prompt text, just like docstrings on
tools were last week.

---
# Part 4 — LCEL: snapping parts together

**LCEL** (LangChain Expression Language) composes Runnables with the `|` pipe: the output
of one becomes the input of the next. A **chain** is just parts piped together.

In [9]:
from langchain_core.output_parsers import StrOutputParser

chain = summarise | llm | StrOutputParser()     # template -> model -> plain string

print(chain.invoke({"audience": "support lead", "words": 10,
                    "review": "Delivered a day late and the box was crushed, but earbuds work fine."}))
print("\nthe chain is itself a Runnable:", [n.name for n in chain.get_graph().nodes.values()])

Late delivery, damaged box, earbuds function well.

the chain is itself a Runnable: ['PromptInput', 'ChatPromptTemplate', 'ChatOpenAI', 'StrOutputParser', 'StrOutputParserOutput']


Because the whole chain is a Runnable, it gets `batch`, `stream` and async for free —
you never write concurrency code yourself.

## The four composition patterns you need

| Pattern | LCEL | What it replaced |
|---|---|---|
| Sequence | `a \| b \| c` | `LLMChain`, `SequentialChain` |
| Parallel fan-out | `RunnableParallel(x=a, y=b)` | `MapReduceChain` (partly) |
| Pass data along | `RunnablePassthrough.assign(new=...)` | chain input/output juggling |
| Branch / route | a function returning the next Runnable | `RouterChain`, `MultiPromptChain` |

In [10]:
from operator import itemgetter
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda

# Parallel: two independent jobs on the same input, run at the same time.
# Every branch receives the SAME input -- here a dict. The prompt template can read a
# dict, but the structured-output model needs text, so that branch first pulls out the
# field it needs. itemgetter("input") | analyst is the standard LCEL idiom for that.
both = RunnableParallel(
    analysis=itemgetter("input") | analyst,
    one_line=ChatPromptTemplate.from_template("Summarise in 8 words: {input}") | llm | StrOutputParser(),
)
out = both.invoke({"input": "Watch strap snapped on day 3. Customer care never replied to 4 emails."})
print("analysis:", out["analysis"].sentiment, out["analysis"].category, out["analysis"].severity)
print("one line:", out["one_line"])

analysis: negative support 4
one line: Watch strap broke day 3; customer care unresponsive.


In [11]:
# assign: keep the original input AND add computed fields to it
enrich = RunnablePassthrough.assign(analysis=lambda x: analyst.invoke(x["review"]))
row = enrich.invoke({"review_id": "DEMO-1", "review": "Great bass, battery barely lasts 3 hours."})
print(row["review_id"], "->", row["analysis"].sentiment, row["analysis"].issues)

DEMO-1 -> mixed ['short battery life']


In [12]:
# Branch: a plain Python function decides which chain runs next.
apology = ChatPromptTemplate.from_template(
    "Write a 2-sentence apology to a customer whose issue is: {issues}. Offer a replacement.") | llm | StrOutputParser()
thanks = ChatPromptTemplate.from_template(
    "Write a 1-sentence thank-you to a happy customer who liked: {what}.") | llm | StrOutputParser()


def route(row):
    a = row["analysis"]
    if a.sentiment == "negative" and a.severity >= 4:
        return RunnableLambda(lambda r: {"issues": ", ".join(r["analysis"].issues)}) | apology
    return RunnableLambda(lambda r: {"what": r["review"]}) | thanks


responder = enrich | RunnableLambda(route)
print(responder.invoke({"review": "Speaker died after 10 days, won't even turn on."}))
print(responder.invoke({"review": "Loud, clear and the battery lasts all weekend!"}))

We’re very sorry to hear that your speaker has stopped working and won’t turn on. Please allow us to send you a replacement right away to resolve this issue.


Thank you so much for your kind words—we’re thrilled to hear you love the loud, clear sound and long-lasting battery!


That's a **router**: rung 2 of Week 10's autonomy ladder. The model classifies; your
code decides what happens next. Deterministic where it matters, flexible where it helps.

---
# Part 5 — Tools, without an agent

A tool is a function plus a schema, exactly as in Week 10 and 11. `@tool` builds the
schema from the docstring and type hints; `bind_tools` hands the schemas to the model.

In [13]:
from langchain.tools import tool

PRICES = {"pulse x1": 2499, "orbit": 3999, "echo bar": 1799}


@tool
def get_price(product: str) -> str:
    """Get the current price in INR of a Nimbus product.

    Args:
        product: Product name, e.g. "Pulse X1", "Orbit" or "Echo Bar".
    """
    for name, price in PRICES.items():
        if name in product.lower():
            return f"{name.title()}: INR {price}"
    return f"Unknown product. Known: {', '.join(n.title() for n in PRICES)}"


with_tools = llm.bind_tools([get_price])
ai = with_tools.invoke("How much do the Pulse X1 earbuds cost?")
print("content   :", repr(ai.content))
print("tool_calls:", ai.tool_calls)

content   : ''
tool_calls: [{'name': 'get_price', 'args': {'product': 'Pulse X1'}, 'id': 'call_DRu26FVqc8TzJf0ZfqEAVCCd', 'type': 'tool_call'}]


Same lesson as Week 10: **the model only asks.** `content` is empty and `tool_calls`
names the function and arguments. Nothing ran. Running the tool, feeding the result back
and looping is the *agent* part — and that's exactly what `create_agent` does, next session.

---
# Part 6 — Project: review intelligence, with no agent

**The job:** a product team has a CSV of customer reviews for three products. They want
every review classified, the top problems per product, a draft reply for every angry
customer, and a one-page report.

**Why no agent?** Every step is known in advance: analyse → aggregate → summarise →
reply → report. Nothing needs the model to *decide what to do next*. That's a
**pipeline**, and it's cheaper, faster and easier to test than an agent. Week 10's rule:
take the lowest rung that solves the problem.

## The data

36 reviews across three (fictional) Nimbus products, in English and Hinglish — the way
Indian customers actually write.

In [14]:
import pandas as pd

REVIEWS = [
    ("R-001", "Pulse X1", 5, "Bass is punchy and the ANC is honestly great for this price. Metro commute is peaceful now."),
    ("R-002", "Pulse X1", 2, "Left bud keeps disconnecting every 10 minutes. Very irritating on calls."),
    ("R-003", "Pulse X1", 1, "Charging case hinge toot gaya in 2 weeks. Bekaar build quality."),
    ("R-004", "Pulse X1", 4, "Sound mast hai, fit bhi comfortable. Battery thodi kam hai, around 4 hours with ANC."),
    ("R-005", "Pulse X1", 2, "Battery drains to 40% in 2 hours with ANC on. Not what was advertised."),
    ("R-006", "Pulse X1", 5, "Paired instantly with my Android, touch controls work smoothly. Value for money."),
    ("R-007", "Pulse X1", 1, "Right earbud stopped charging after 3 weeks. Support asked me to courier it myself at my cost."),
    ("R-008", "Pulse X1", 3, "Good sound but ears start paining after an hour. Tips are too hard."),
    ("R-009", "Pulse X1", 2, "Mic quality is terrible, people can't hear me on Zoom calls."),
    ("R-010", "Pulse X1", 4, "Great for gym, sweat-proof as promised. Wish the case was smaller."),
    ("R-011", "Pulse X1", 1, "Connection drops whenever phone is in my pocket. Returned it."),
    ("R-012", "Pulse X1", 5, "Delivered next day in Pune, sound is crisp. Loving it."),
    ("R-013", "Orbit", 5, "Step count accurate, sleep tracking is surprisingly good. Battery lasts 9 days!"),
    ("R-014", "Orbit", 1, "App logs me out every day and I lose my workout data. Useless."),
    ("R-015", "Orbit", 2, "Strap broke on day 5. Customer care never replied to 4 emails."),
    ("R-016", "Orbit", 4, "Display is bright even in Chennai sun. Heart rate a little off during running."),
    ("R-017", "Orbit", 1, "Watch got very hot while charging, smelled of burning plastic. Scared to use it now."),
    ("R-018", "Orbit", 3, "Decent watch but notifications come late, sometimes 5 minutes late."),
    ("R-019", "Orbit", 2, "App sync fails with iPhone. Android pe theek chalta hai, iOS pe bilkul nahi."),
    ("R-020", "Orbit", 5, "SpO2 and stress tracking useful for my parents. Easy to set up."),
    ("R-021", "Orbit", 4, "Looks premium, battery 7 days. The app UI is confusing though."),
    ("R-022", "Orbit", 1, "Delivered with a cracked screen. Replacement taking 3 weeks already."),
    ("R-023", "Orbit", 3, "GPS takes ages to lock. Fine for walking, bad for cycling."),
    ("R-024", "Orbit", 5, "Best watch under 4000. Calling feature works well."),
    ("R-025", "Echo Bar", 5, "Loud enough for a house party. Bass is solid for the size."),
    ("R-026", "Echo Bar", 2, "Bluetooth range is very short, disconnects if I go to the next room."),
    ("R-027", "Echo Bar", 4, "Good sound for 1799. Battery lasts a full day of music."),
    ("R-028", "Echo Bar", 1, "Stopped working after it got a little wet, even though it says water resistant."),
    ("R-029", "Echo Bar", 3, "Sound is okay but distorts at full volume. Theek hai price ke hisaab se."),
    ("R-030", "Echo Bar", 5, "Perfect for my small room, pairing is instant."),
    ("R-031", "Echo Bar", 2, "Delivery took 12 days to Guwahati and the box was damaged."),
    ("R-032", "Echo Bar", 4, "Compact, good battery. Wish it had an aux input."),
    ("R-033", "Echo Bar", 1, "Charging port became loose within a month. Now it doesn't charge at all."),
    ("R-034", "Echo Bar", 5, "Gifted to my brother, he loves it. Value for money."),
    ("R-035", "Echo Bar", 2, "Support chat bot kept looping, never got a human. Speaker has a crackling noise."),
    ("R-036", "Echo Bar", 4, "Clear vocals, great for podcasts. Bass could be deeper."),
]
df = pd.DataFrame(REVIEWS, columns=["review_id", "product", "stars", "text"])
df.to_csv("reviews.csv", index=False)
print(df.shape)
df.groupby("product")["stars"].agg(["count", "mean"]).round(2)

(36, 4)


,count,mean
product,,
Echo Bar,12,3.17
Orbit,12,3.00
Pulse X1,12,2.92


## Step A — Analyse every review (batch, in parallel)

One structured-output chain, run over all 36 reviews with `batch`. `max_concurrency`
sends several requests at once — the same job as a `for` loop, several times faster.

In [15]:
analyse_prompt = ChatPromptTemplate.from_messages([
    ("system", "You analyse customer reviews of consumer electronics sold in India. "
               "Reviews may be in English or Hinglish. Be precise about issues."),
    ("human", "Product: {product}\nStars: {stars}\nReview: {text}"),
])
analyse_chain = analyse_prompt | llm.with_structured_output(ReviewAnalysis)

t0 = time.time()
analyses = analyse_chain.batch(df[["product", "stars", "text"]].to_dict("records"),
                               config={"max_concurrency": 8})
print(f"{len(analyses)} reviews analysed in {time.time() - t0:.1f}s")

df["sentiment"] = [a.sentiment for a in analyses]
df["category"] = [a.category for a in analyses]
df["severity"] = [a.severity for a in analyses]
df["language"] = [a.language for a in analyses]
df["issues"] = [a.issues for a in analyses]
df[["review_id", "product", "stars", "sentiment", "category", "severity", "language"]].head(12)

36 reviews analysed in 4.5s


,review_id,product,stars,sentiment,category,severity,language
0,R-001,Pulse X1,5,positive,sound,1,english
1,R-002,Pulse X1,2,negative,connectivity,4,english
2,R-003,Pulse X1,1,negative,build quality,4,hinglish
3,R-004,Pulse X1,4,positive,battery,2,hinglish
4,R-005,Pulse X1,2,negative,battery,4,english
5,R-006,Pulse X1,5,positive,connectivity,1,english
6,R-007,Pulse X1,1,negative,support,4,english
7,R-008,Pulse X1,3,mixed,comfort,3,english
8,R-009,Pulse X1,2,negative,sound,4,english
9,R-010,Pulse X1,4,positive,comfort,2,english


## Step B — Aggregate with pandas, not with the model

Counting, grouping and averaging are **code** jobs. Don't pay a model to do arithmetic
it might get wrong.

In [16]:
print("Sentiment by product:")
print(pd.crosstab(df["product"], df["sentiment"]), "\n")

negative = df[df["sentiment"] != "positive"]
print("Top problem categories (non-positive reviews):")
print(negative.groupby(["product", "category"]).size().sort_values(ascending=False).head(8), "\n")

print("Languages:", df["language"].value_counts().to_dict())
print("Severity 5 (urgent):", df.loc[df["severity"] == 5, "review_id"].tolist())

Sentiment by product:
sentiment  mixed  negative  positive
product                             
Echo Bar       1         5         6
Orbit          2         7         3
Pulse X1       1         6         5 

Top problem categories (non-positive reviews):
product   category     
Orbit     app              3
Echo Bar  build quality    2
Pulse X1  connectivity     2
Orbit     connectivity     2
          support          1
Pulse X1  sound            1
          comfort          1
          build quality    1
dtype: int64 

Languages: {'english': 32, 'hinglish': 4}
Severity 5 (urgent): ['R-017']


## Step C — Summarise each product's problems (parallel)

Now the model earns its keep: turning a pile of issue phrases into a readable diagnosis.
One chain, run once per product, in parallel.

> ### ⚠️ What went wrong in our first draft
> The first version of this prompt asked the model to say *how many reviews* mentioned each
> problem. For a product with **12 reviews** it wrote *"mentioned in 120 reviews"* — and the
> final report repeated the invented numbers with total confidence.
>
> **Counting is code.** pandas counts; the model describes. The prompt below hands the
> model exact counts and forbids it from inventing any.

In [17]:
theme_prompt = ChatPromptTemplate.from_template(
    "You are a product-quality analyst for the {product}.\n\n"
    "Complaint counts by category, from unhappy reviews (computed by code; exact):\n{counts}\n\n"
    "What those customers actually said:\n{issues}\n\n"
    "Write 3 bullet points: the biggest problems, most frequent first. Quote counts ONLY from "
    "the table above and never invent a number. Then one line: 'Fix first:' and the single "
    "highest-impact fix."
)
theme_chain = theme_prompt | llm | StrOutputParser()

inputs = []
for product, group in df.groupby("product"):
    unhappy = group[group["sentiment"] != "positive"]
    counts = unhappy["category"].value_counts()
    issues = [i for lst in unhappy["issues"] for i in lst]
    inputs.append({"product": product,
                   "counts": "\n".join(f"- {cat}: {n} review(s)" for cat, n in counts.items()),
                   "issues": "\n".join(f"- {i}" for i in issues)})

themes = dict(zip([i["product"] for i in inputs], theme_chain.batch(inputs)))
for product, text in themes.items():
    print(f"=== {product} ===\n{text}\n")

=== Echo Bar ===
- Build quality issues are the most frequent complaint, with 2 reviews citing problems such as a loose charging port, device not charging, and lack of water resistance.  
- Connectivity problems, including short Bluetooth range and frequent disconnections, were mentioned in 1 review.  
- Sound quality concerns, including distortion at full volume and crackling noise, appeared in 1 review.  

Fix first: Improve build quality to ensure reliable charging and water resistance.

=== Orbit ===
- App issues are the most frequent complaint, with 3 reviews citing problems such as daily logouts, sync failures on iOS, and confusing UI.  
- Connectivity problems follow, with 2 reviews mentioning slow GPS lock and late notifications.  
- Other notable issues include support delays (1 review), battery and overheating concerns (1 review), and physical product defects like strap breakage and cracked screens (1 review each).  

Fix first: Improve app stability and usability to address 

## Step D — Draft replies, routed by severity

The router from Part 4, applied to real data: angry customers get an apology with a
concrete next step; the rest get nothing (don't spam happy customers). A **safety** issue
— overheating, burning smell — gets escalated instead of a template apology.

In [18]:
reply_prompt = ChatPromptTemplate.from_template(
    "Write a short, warm reply (max 60 words) from Nimbus support to this customer. "
    "Acknowledge the specific problem, apologise once, and offer a concrete next step "
    "(free pickup for replacement). Match their language: reply in Hinglish if they wrote Hinglish.\n\n"
    "Product: {product}\nTheir review: {text}"
)
reply_chain = reply_prompt | llm | StrOutputParser()

import re

# Word boundaries matter: a bare "hot" would also match "photos" and "shot".
SAFETY = re.compile(r"\b(hot|overheat\w*|burn\w*|smoke|smoking|fire|melt\w*|spark\w*)\b", re.I)


def triage(row):
    if SAFETY.search(row["text"]):
        return "ESCALATE: safety"
    if row["sentiment"] == "negative" and row["severity"] >= 4:
        return "reply"
    return "no action"


df["action"] = df.apply(triage, axis=1)
to_reply = df[df["action"] == "reply"]
replies = reply_chain.batch(to_reply[["product", "text"]].to_dict("records"), config={"max_concurrency": 8})
df.loc[to_reply.index, "draft_reply"] = replies

print(df["action"].value_counts().to_dict(), "\n")
for _, r in df[df["action"] != "no action"].head(4).iterrows():
    print(f"{r.review_id} [{r.action}] {r.text[:70]}")
    if isinstance(r.get("draft_reply"), str):
        print(f"   -> {r.draft_reply}\n")
    else:
        print("   -> routed to the safety team, no auto-reply\n")

{'no action': 24, 'reply': 11, 'ESCALATE: safety': 1} 

R-002 [reply] Left bud keeps disconnecting every 10 minutes. Very irritating on call
   -> Hi! Sorry to hear your Pulse X1 left bud keeps disconnecting—definitely frustrating during calls. Hum aapke liye free pickup arrange kar dete hain taaki hum aapka replacement bhej saken. Please share your order details, and we’ll get this sorted ASAP!

R-003 [reply] Charging case hinge toot gaya in 2 weeks. Bekaar build quality.
   -> Hi! Sorry to hear about the Pulse X1 charging case hinge issue. Yeh bilkul expected nahi hai. Hum aapke liye free pickup arrange kar dete hain aur replacement case bhejte hain. Please DM your order details, hum turant help karenge!

R-005 [reply] Battery drains to 40% in 2 hours with ANC on. Not what was advertised.
   -> Hi! Sorry to hear your Pulse X1 battery drains quickly with ANC on. That’s definitely not how it should be. Hum aapke liye free pickup arrange kar dete hain for a replacement. Please share you

Notice the safety check is a **keyword rule, not the model**. For something that must
never be missed, a deterministic check you can read beats a classifier you have to trust.
(Week 11's guardrail lesson, applied to a pipeline.)

## Step E — The report

Everything so far feeds one last chain, which writes the page the product team actually
reads.

In [19]:
report_prompt = ChatPromptTemplate.from_template(
    "Write a one-page markdown report for the Nimbus product team.\n\n"
    "Stats (computed, trust these exactly):\n{stats}\n\n"
    "Problem themes per product:\n{themes}\n\n"
    "Sections: '## Headline' (2 sentences), '## Per product' (3 bullets each), "
    "'## Do this week' (3 numbered actions, most urgent first). Mention the safety escalations first. "
    "Use only numbers given above. Return plain markdown, not wrapped in a code block."
)
report_chain = report_prompt | llm | StrOutputParser()

stats = {
    "reviews": len(df),
    "by_product": df.groupby("product")["stars"].mean().round(2).to_dict(),
    "sentiment": pd.crosstab(df["product"], df["sentiment"]).to_dict(),
    "safety_escalations": df.loc[df["action"] == "ESCALATE: safety", ["review_id", "product"]].values.tolist(),
    "replies_drafted": int((df["action"] == "reply").sum()),
    "hinglish_share": f"{(df['language'] == 'hinglish').mean():.0%}",
}
report = report_chain.invoke({"stats": json.dumps(stats, indent=1),
                              "themes": "\n\n".join(f"{p}:\n{t}" for p, t in themes.items())})
Path("review_report.md").write_text(report)
df.to_csv("reviews_analysed.csv", index=False)
print(report)

## Headline

The Nimbus product team has received 36 reviews across three products, with an overall mixed sentiment and 11 drafted replies. A critical safety escalation (R-017) was reported for the Orbit, requiring immediate attention.

## Per product

- **Echo Bar**  
  - Average rating: 3.17 from 12 reviews (6 positive, 1 mixed, 5 negative).  
  - Main issues: build quality problems including loose charging port, device not charging, and lack of water resistance (2 reviews). Connectivity and sound quality concerns were also noted.  
  - Priority fix: enhance build quality to ensure reliable charging and water resistance.

- **Orbit**  
  - Average rating: 3.0 from 12 reviews (3 positive, 2 mixed, 7 negative).  
  - Key problems: app instability with daily logouts, sync failures on iOS, and confusing UI (3 reviews); connectivity issues like slow GPS lock and late notifications (2 reviews); plus support delays, battery/overheating, and physical defects.  
  - Priority fix: improve app 

## What we built

```
reviews.csv ─► analyse (batch ×36) ─► pandas stats ─► themes (parallel ×3) ─► report
                                   └─► triage rule ─► replies (batch) / safety escalation
```

- **Five chains**, each a template → model → parser. No agent, no loop.
- **The model did language work** (classify, summarise, write). **Code did the rest**
  (count, group, route, the safety rule).
- Every step is testable on its own, and the whole thing costs a few cents.

> **Next session:** when the steps *can't* be written down in advance — "why did Orbit's
> rating drop in March, and which city is worst?" — you need a model that decides what to
> do next. That's `create_agent`.

---
## Exercises

**1. Swap the model.** Change `init_chat_model` to `"openai:gpt-4.1-nano"` and rerun Step A.
Compare the categories with the mini model's. Where does the cheaper model disagree?

**2. Add a field.** Add `mentions_competitor: bool` to `ReviewAnalysis`. Nothing else
changes — why?

**3. Stream the report.** Replace `report_chain.invoke` with `report_chain.stream` and
print chunks as they arrive.

**4. A second language.** Add three reviews in Hindi (Devanagari). Does the analysis cope?
Does the reply come back in Hindi?

**5. Test a chain.** Write 5 reviews with known correct answers and assert the sentiment
of each. That's a unit test for a chain — and it's why pipelines are easier to ship than
agents.